# Constrained Optimization — Theory Notebook

> *"The world is full of constraints. The art of optimization is finding the lowest point you're actually allowed to visit."*

Interactive demonstrations of KKT conditions, projected GD, penalty methods, barrier methods, ADMM, and the SVM dual. Companion to [notes.md](notes.md).

In [ ]:
import numpy as np
import scipy.linalg as la
import scipy.optimize as opt

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    try:
        import seaborn as sns
        sns.set_theme(style='whitegrid', palette='colorblind')
        HAS_SNS = True
    except ImportError:
        plt.style.use('seaborn-v0_8-whitegrid')
        HAS_SNS = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

if HAS_MPL:
    mpl.rcParams.update({
        'figure.figsize': (10, 6),
        'figure.dpi': 120,
        'font.size': 13,
        'axes.titlesize': 15,
        'axes.labelsize': 13,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'lines.linewidth': 2.0,
        'axes.spines.top': False,
        'axes.spines.right': False,
    })

np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)
print('Setup complete.')

---

## 1. Lagrange Multipliers for Equality Constraints

Solve $\min \frac{1}{2}\lVert \mathbf{x} \rVert_2^2$ subject to $A\mathbf{x} = \mathbf{b}$.

In [ ]:
# === 1.1 Minimum-norm solution ===
A = np.array([[1.0, 1.0, 0.0], [0.0, 1.0, 1.0]])
b = np.array([1.0, 2.0])

# Closed-form: x* = A^T (A A^T)^{-1} b
x_star = A.T @ la.solve(A @ A.T, b)

print(f'A = \n{A}')
print(f'b = {b}')
print(f'\nMinimum-norm solution: x* = {x_star}')
print(f'Verify Ax = b: A @ x* = {A @ x_star}')
print(f'||x*|| = {np.linalg.norm(x_star):.6f}')

# Verify using KKT system
K = np.block([[np.eye(3), A.T], [A, np.zeros((2, 2))]])
rhs = np.concatenate([np.zeros(3), b])
sol = la.solve(K, rhs)
x_kkt = sol[:3]
nu_kkt = sol[3:]

print(f'\nKKT solution: x = {x_kkt}')
print(f'Lagrange multipliers: nu = {nu_kkt}')

ok = np.allclose(x_star, x_kkt)
print(f"\n{'PASS' if ok else 'FAIL'} — closed-form matches KKT solution")

---

## 2. KKT Conditions: A Complete Example

Solve $\min x_1^2 + x_2^2$ subject to $x_1 + x_2 \geq 1$.

In [ ]:
# === 2.1 KKT analysis ===
# Problem: min x1^2 + x2^2 s.t. 1 - x1 - x2 <= 0
# Lagrangian: L = x1^2 + x2^2 + lambda*(1 - x1 - x2)
# KKT: 2*x1 - lambda = 0, 2*x2 - lambda = 0, lambda >= 0, lambda*(1-x1-x2) = 0

# Case 1: lambda = 0 => x1 = x2 = 0, but 1-0-0 = 1 > 0 (infeasible)
# Case 2: 1 - x1 - x2 = 0 => x1 = x2 = lambda/2, so lambda = 1, x1 = x2 = 0.5

x1, x2 = 0.5, 0.5
lam = 1.0

print('KKT conditions verification:')
print(f'  Stationarity: 2*x1 - lambda = {2*x1 - lam:.6f} (should be 0)')
print(f'  Stationarity: 2*x2 - lambda = {2*x2 - lam:.6f} (should be 0)')
print(f'  Primal feasibility: 1 - x1 - x2 = {1 - x1 - x2:.6f} (should be <= 0)')
print(f'  Dual feasibility: lambda = {lam:.6f} (should be >= 0)')
print(f'  Complementary slackness: lambda*(1-x1-x2) = {lam*(1-x1-x2):.6f} (should be 0)')

ok = (abs(2*x1 - lam) < 1e-10 and abs(2*x2 - lam) < 1e-10 and
      1 - x1 - x2 <= 1e-10 and lam >= 0 and abs(lam*(1-x1-x2)) < 1e-10)
print(f"\n{'PASS' if ok else 'FAIL'} — all KKT conditions satisfied")

---

## 3. Projected Gradient Descent

Minimize $f(\mathbf{x}) = \frac{1}{2}\lVert A\mathbf{x} - \mathbf{b} \rVert_2^2$ subject to $\mathbf{x} \geq \mathbf{0}$.

In [ ]:
# === 3.1 Non-negative least squares ===
np.random.seed(42)
m, n = 100, 20
A = np.random.randn(m, n)
x_true = np.abs(np.random.randn(n))  # True solution is non-negative
b = A @ x_true + 0.1 * np.random.randn(m)

def f_nnls(x): return 0.5 * np.linalg.norm(A @ x - b)**2
def grad_nnls(x): return A.T @ (A @ x - b)

def proj_nonnegative(x):
    return np.maximum(x, 0)

# Projected GD
L = np.linalg.norm(A.T @ A)  # Smoothness constant
eta = 1.0 / L
x = np.zeros(n)
T = 1000
obj_hist = []

for t in range(T):
    g = grad_nnls(x)
    x = proj_nonnegative(x - eta * g)
    obj_hist.append(f_nnls(x))

# Compare with scipy
res = opt.nnls(A, b)
x_scipy = res[0]

print(f'Projected GD after {T} iterations:')
print(f'  Objective: {obj_hist[-1]:.6f}')
print(f'  ||x - x_true||: {np.linalg.norm(x - x_true):.6f}')
print(f'  Min(x): {x.min():.6f} (should be >= 0)')
print(f'\nscipy nnls:')
print(f'  Objective: {0.5 * np.linalg.norm(A @ x_scipy - b)**2:.6f}')
print(f'  ||x_pg - x_scipy||: {np.linalg.norm(x - x_scipy):.6f}')

ok = np.linalg.norm(x - x_scipy) < 0.1
print(f"\n{'PASS' if ok else 'FAIL'} — projected GD matches scipy nnls")

In [ ]:
# === 3.2 Plot convergence ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(obj_hist, color=COLORS['primary'])
    axes[0].set_title('Projected GD: objective vs iteration')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Objective')
    
    axes[1].bar(range(n), x, color=COLORS['primary'], alpha=0.7)
    axes[1].axhline(0, color=COLORS['neutral'], linestyle='--')
    axes[1].set_title('Non-negative solution')
    axes[1].set_xlabel('Index')
    axes[1].set_ylabel('x_i')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 4. Projection onto the Probability Simplex

Project a vector onto $\Delta_n = \{\mathbf{x} : \mathbf{x} \geq \mathbf{0}, \mathbf{1}^\top \mathbf{x} = 1\}$.

In [ ]:
# === 4.1 Simplex projection algorithm ===
def proj_simplex(v):
    """Project v onto the probability simplex. O(n log n)."""
    n = len(v)
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u) - 1
    ind = np.arange(n) + 1
    cond = u - cssv / ind > 0
    rho = ind[cond][-1]
    theta = cssv[cond][-1] / rho
    return np.maximum(v - theta, 0)

# Test
v = np.array([0.5, -0.3, 0.8, 0.2, -0.1])
x_proj = proj_simplex(v)

print(f'Input: {v}')
print(f'Projected: {x_proj}')
print(f'Sum: {x_proj.sum():.10f} (should be 1.0)')
print(f'Min: {x_proj.min():.10f} (should be >= 0)')

ok = abs(x_proj.sum() - 1.0) < 1e-10 and x_proj.min() >= -1e-10
print(f"\n{'PASS' if ok else 'FAIL'} — projection onto simplex is correct")

In [ ]:
# === 4.2 Visualize projection ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].bar(range(len(v)), v, color=COLORS['secondary'], alpha=0.7)
    axes[0].axhline(0, color=COLORS['neutral'], linestyle='--')
    axes[0].set_title('Input vector (not on simplex)')
    axes[0].set_xlabel('Index')
    axes[0].set_ylabel('Value')
    
    axes[1].bar(range(len(x_proj)), x_proj, color=COLORS['primary'], alpha=0.7)
    axes[1].axhline(0, color=COLORS['neutral'], linestyle='--')
    axes[1].set_title('Projected onto simplex')
    axes[1].set_xlabel('Index')
    axes[1].set_ylabel('Value')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 5. Penalty Methods

Solve $\min x^2$ subject to $x \geq 1$ using the quadratic penalty method.

In [ ]:
# === 5.1 Quadratic penalty method ===
def f_pen(x, mu):
    return x**2 + mu * max(0, 1 - x)**2

def grad_f_pen(x, mu):
    if x >= 1:
        return 2*x
    return 2*x - 2*mu*(1 - x)

mu_values = [1, 10, 100, 1000, 10000]
solutions = []

for mu in mu_values:
    res = opt.minimize_scalar(lambda x: f_pen(x, mu), bounds=(-5, 5), method='bounded')
    solutions.append((mu, res.x, res.fun))

print('Quadratic penalty method for min x^2 s.t. x >= 1:')
print(f'{'mu':>8} | {'x(mu)':>10} | {'f(x)':>10} | {'Violation':>10}')
print('-' * 50)
for mu, x_mu, f_val in solutions:
    violation = max(0, 1 - x_mu)
    print(f'{mu:>8} | {x_mu:>10.6f} | {f_val:>10.6f} | {violation:>10.6f}')

ok = abs(solutions[-1][1] - 1.0) < 0.01
print(f"\n{'PASS' if ok else 'FAIL'} — solution converges to x* = 1")

---

## 6. Barrier (Interior-Point) Methods

Solve $\min x^2$ subject to $x \geq 1$ using the logarithmic barrier.

In [ ]:
# === 6.1 Logarithmic barrier ===
def f_barrier(x, t):
    if x <= 1:
        return np.inf
    return t * x**2 - np.log(x - 1)

def grad_f_barrier(x, t):
    if x <= 1:
        return np.inf
    return 2*t*x - 1/(x - 1)

t_values = [1, 10, 100, 1000]
solutions = []

for t in t_values:
    res = opt.minimize_scalar(lambda x: f_barrier(x, t), bounds=(1.001, 10), method='bounded')
    solutions.append((t, res.x, res.fun/t))

print('Logarithmic barrier method for min x^2 s.t. x >= 1:')
print(f'{'t':>8} | {'x*(t)':>10} | {'f(x)':>10} | {'Duality gap ~1/t':>15}')
print('-' * 55)
for t, x_t, f_val in solutions:
    print(f'{t:>8} | {x_t:>10.6f} | {f_val:>10.6f} | {1/t:>15.6f}')

ok = abs(solutions[-1][1] - 1.0) < 0.01
print(f"\n{'PASS' if ok else 'FAIL'} — solution converges to x* = 1 from interior")

---

## 7. SVM Dual Problem

Derive and solve the SVM dual problem.

In [ ]:
# SVM dual solver
np.random.seed(42)
n_data = 100
X_pos = np.random.randn(n_data // 2, 2) + np.array([2, 2])
X_neg = np.random.randn(n_data // 2, 2) + np.array([-2, -2])
X_svm = np.vstack([X_pos, X_neg])
y_svm = np.concatenate([np.ones(n_data // 2), -np.ones(n_data // 2)])

K = X_svm @ X_svm.T
C = 1.0
P = np.outer(y_svm, y_svm) * K
q = -np.ones(n_data)
bounds = [(0, C) for _ in range(n_data)]


def dual_obj(alpha):
    return 0.5 * alpha @ P @ alpha - np.sum(alpha)


def dual_grad(alpha):
    return P @ alpha - 1.0


alpha0 = np.zeros(n_data)
constraints = {'type': 'eq', 'fun': lambda a: float(np.dot(y_svm, a)), 'jac': lambda a: y_svm.copy()}
res = opt.minimize(
    dual_obj,
    alpha0,
    jac=dual_grad,
    bounds=bounds,
    constraints=constraints,
    method='SLSQP',
    options={'maxiter': 1000, 'ftol': 1e-12},
)
alpha = res.x
sv_idx = alpha > 1e-5
w = np.sum(alpha[sv_idx, None] * y_svm[sv_idx, None] * X_svm[sv_idx], axis=0)
margin_sv = (alpha > 1e-5) & (alpha < C - 1e-5)
if np.any(margin_sv):
    b = np.mean(y_svm[margin_sv] - X_svm[margin_sv] @ w)
else:
    b = np.mean(y_svm[sv_idx] - X_svm[sv_idx] @ w)
print('SVM dual solution:')
print(f'Optimization success: {res.success}')
print(f'Number of support vectors: {np.sum(sv_idx)}')
print(f'w = {w}')
print(f'b = {b:.6f}')
print(f'Dual objective: {-res.fun:.6f}')
ok = res.success and 0 < np.sum(sv_idx) < n_data
print('PASS' if ok else 'FAIL', '- found a non-degenerate support-vector set')

In [ ]:
# === 7.2 Plot SVM decision boundary ===
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Plot data
    ax.scatter(X_svm[y_svm == 1, 0], X_svm[y_svm == 1, 1],
               color=COLORS['primary'], s=40, label='Class +1')
    ax.scatter(X_svm[y_svm == -1, 0], X_svm[y_svm == -1, 1],
               color=COLORS['secondary'], s=40, label='Class -1')
    
    # Highlight support vectors
    ax.scatter(X_svm[sv_idx, 0], X_svm[sv_idx, 1],
               facecolors='none', edgecolors=COLORS['error'], s=120, linewidth=2,
               label='Support vectors')
    
    # Decision boundary
    x1_range = np.linspace(-5, 5, 100)
    x2_boundary = -(w[0] * x1_range + b) / w[1]
    ax.plot(x1_range, x2_boundary, color=COLORS['error'], linewidth=2, label='Decision boundary')
    
    # Margin boundaries
    margin = 1.0 / np.linalg.norm(w)
    x2_upper = -(w[0] * x1_range + b - 1) / w[1]
    x2_lower = -(w[0] * x1_range + b + 1) / w[1]
    ax.plot(x1_range, x2_upper, '--', color=COLORS['neutral'], alpha=0.5, label='Margin')
    ax.plot(x1_range, x2_lower, '--', color=COLORS['neutral'], alpha=0.5)
    
    ax.set_title('SVM: Decision boundary and support vectors')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.legend()
    ax.set_aspect('equal')
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 8. ADMM for Lasso

Solve $\min \frac{1}{2}\lVert A\mathbf{x} - \mathbf{b} \rVert_2^2 + \lambda \lVert \mathbf{x} \rVert_1$ using ADMM.

In [ ]:
# === 8.1 ADMM Lasso solver ===
def soft_threshold(x, threshold):
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)

def admm_lasso(A, b, lam, rho=1.0, max_iter=1000, tol=1e-6):
    n, p = A.shape
    x = np.zeros(p)
    z = np.zeros(p)
    u = np.zeros(p)
    
    # Precompute Cholesky factorization
    AtA = A.T @ A
    L = np.linalg.cholesky(AtA + rho * np.eye(p))
    
    primal_res = []
    dual_res = []
    obj_hist = []
    
    for k in range(max_iter):
        # x-update
        rhs = A.T @ b + rho * (z - u)
        x = np.linalg.solve(L.T, np.linalg.solve(L, rhs))
        
        # z-update (soft thresholding)
        z_old = z.copy()
        z = soft_threshold(x + u, lam / rho)
        
        # u-update
        u = u + x - z
        
        # Residuals
        primal_res.append(np.linalg.norm(x - z))
        dual_res.append(rho * np.linalg.norm(z - z_old))
        obj_hist.append(0.5 * np.linalg.norm(A @ x - b)**2 + lam * np.linalg.norm(z, 1))
        
        if primal_res[-1] < tol and dual_res[-1] < tol:
            break
    
    return x, np.array(primal_res), np.array(dual_res), np.array(obj_hist)

np.random.seed(42)
m, p = 100, 50
A_lasso = np.random.randn(m, p)
x_true = np.zeros(p)
x_true[:10] = np.array([1.0, -0.5, 0.8, -0.3, 0.6, 0.2, -0.4, 0.7, -0.1, 0.3])
b_lasso = A_lasso @ x_true + 0.1 * np.random.randn(m)

lam = 0.5
x_admm, pres, dres, obj = admm_lasso(A_lasso, b_lasso, lam)

print(f'ADMM Lasso: m={m}, p={p}, lambda={lam}')
print(f'Iterations: {len(obj)}')
print(f'Final objective: {obj[-1]:.6f}')
print(f'Final primal residual: {pres[-1]:.2e}')
print(f'Final dual residual: {dres[-1]:.2e}')
print(f'Number of non-zeros: {np.sum(np.abs(x_admm) > 1e-6)}')
print(f'||x_admm - x_true||: {np.linalg.norm(x_admm - x_true):.6f}')

ok = pres[-1] < 1e-4 and dres[-1] < 1e-4
print(f"\n{'PASS' if ok else 'FAIL'} — ADMM converged")

In [ ]:
# === 8.2 Plot ADMM convergence ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].plot(pres, color=COLORS['primary'], label='Primal')
    axes[0].plot(dres, color=COLORS['secondary'], label='Dual')
    axes[0].set_yscale('log')
    axes[0].set_title('ADMM residuals')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Residual norm')
    axes[0].legend()
    
    axes[1].plot(obj, color=COLORS['primary'])
    axes[1].set_title('Objective value')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Objective')
    
    axes[2].bar(range(p), x_admm, color=COLORS['primary'], alpha=0.7)
    axes[2].bar(range(p), x_true, color=COLORS['secondary'], alpha=0.5)
    axes[2].set_title('Solution vs True')
    axes[2].set_xlabel('Index')
    axes[2].set_ylabel('Value')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 9. Portfolio Optimization with Constraints

Mean-variance portfolio optimization with budget and return constraints.

In [ ]:
# === 9.1 Constrained portfolio optimization ===
np.random.seed(42)
n_assets = 10
mu_port = np.random.uniform(0.05, 0.15, n_assets)  # Expected returns
Sigma = np.random.randn(n_assets, n_assets)
Sigma = Sigma @ Sigma.T / n_assets + 0.01 * np.eye(n_assets)  # Covariance

r_target = 0.10  # Target return

    # min 0.5 * w^T Sigma w s.t. mu^T w = r_target, 1^T w = 1
# KKT system:
# [Sigma  mu  1] [w]   [0]
# [mu^T  0   0] [v1] = [r_target]
# [1^T   0   0] [v2]   [1]

K_port = np.block([
    [Sigma, mu_port.reshape(-1, 1), np.ones((n_assets, 1))],
    [mu_port.reshape(1, -1), np.zeros((1, 2))],
    [np.ones((1, n_assets)), np.zeros((1, 2))]
])
rhs_port = np.concatenate([np.zeros(n_assets), [r_target, 1.0]])

sol_port = la.solve(K_port, rhs_port)
w_port = sol_port[:n_assets]
nu_port = sol_port[n_assets:]

print(f'Mean-variance portfolio optimization:')
print(f'Number of assets: {n_assets}')
print(f'Target return: {r_target}')
print(f'\nOptimal weights:')
for i, (wi, mui) in enumerate(zip(w_port, mu_port)):
    print(f'  Asset {i+1}: w = {wi:.4f}, mu = {mui:.4f}')
print(f'\nPortfolio return: {w_port @ mu_port:.6f}')
print(f'Portfolio risk: {np.sqrt(w_port @ Sigma @ w_port):.6f}')
print(f'Budget constraint: {w_port.sum():.10f}')

ok = abs(w_port.sum() - 1.0) < 1e-8 and abs(w_port @ mu_port - r_target) < 1e-8
print(f"\n{'PASS' if ok else 'FAIL'} — constraints satisfied")

---

## 10. Trust Region Subproblem

Solve $\min \mathbf{g}^\top \mathbf{d} + \frac{1}{2}\mathbf{d}^\top B \mathbf{d}$ s.t. $\lVert \mathbf{d} \rVert_2 \leq \Delta$.

In [ ]:
# === 10.1 Trust region solver ===
np.random.seed(42)
n = 20
B = np.random.randn(n, n)
B = B.T @ B  # Positive definite
g = np.random.randn(n)
Delta = 1.0

# Solve using scipy
def tr_obj(d): return g @ d + 0.5 * d @ B @ d
def tr_grad(d): return B @ d + g

res = opt.minimize(tr_obj, np.zeros(n), jac=tr_grad,
                   constraints={'type': 'ineq', 'fun': lambda d: Delta - np.linalg.norm(d)},
                   method='SLSQP')

d_opt = res.x

print(f'Trust region subproblem: n={n}, Delta={Delta}')
print(f'||d_opt|| = {np.linalg.norm(d_opt):.6f} (should be <= {Delta})')
print(f'Objective: {tr_obj(d_opt):.6f}')
print(f'Unconstrained min: d = {-la.solve(B, g)}, ||d|| = {np.linalg.norm(la.solve(B, g)):.6f}')

# Check if unconstrained minimum is inside trust region
d_unc = -la.solve(B, g)
if np.linalg.norm(d_unc) <= Delta:
    print('Unconstrained minimum is inside trust region')
    ok = np.allclose(d_opt, d_unc, atol=1e-4)
else:
    print('Unconstrained minimum is outside trust region')
    ok = abs(np.linalg.norm(d_opt) - Delta) < 1e-4

print(f"\n{'PASS' if ok else 'FAIL'} — trust region solution is correct")

---

## Summary

This notebook demonstrated:

1. **Lagrange multipliers** — minimum-norm solution via KKT system
2. **KKT conditions** — complete verification on a simple QP
3. **Projected gradient descent** — non-negative least squares
4. **Simplex projection** — $O(n \log n)$ algorithm
5. **Penalty methods** — quadratic penalty convergence
6. **Barrier methods** — logarithmic barrier from interior
7. **SVM dual** — deriving and solving the dual problem
8. **ADMM** — Lasso via alternating direction method
9. **Portfolio optimization** — mean-variance with constraints
10. **Trust region** — solving the trust region subproblem

See [exercises.ipynb](exercises.ipynb) for graded problems on these topics.

---

## 11. Projected GD on Various Constraint Sets

Compare projected gradient descent on different feasible sets.

In [ ]:
# Comparison of projections
def proj_l2_ball(x, radius=1.0):
    norm = np.linalg.norm(x)
    if norm <= radius:
        return x
    return radius * x / norm


def proj_box(x, lower, upper):
    return np.clip(x, lower, upper)


v = np.array([2.0, -1.5, 0.5, -0.3, 1.8])
p_l2 = proj_l2_ball(v)
p_nn = np.maximum(v, 0)
p_box = proj_box(v, -1, 1)
p_simplex = proj_simplex(v)
print(f'Input vector: {v}')
print(f'L2-ball projection: {p_l2}')
print(f'Nonnegative projection: {p_nn}')
print(f'Box projection: {p_box}')
print(f'Simplex projection: {p_simplex}')
ok = (
    np.linalg.norm(p_l2) <= 1 + 1e-10
    and p_nn.min() >= -1e-10
    and np.all((p_box >= -1 - 1e-10) & (p_box <= 1 + 1e-10))
    and abs(p_simplex.sum() - 1.0) < 1e-10
    and p_simplex.min() >= -1e-10
)
print('PASS' if ok else 'FAIL', '- all projections are correct')

---

## 12. Augmented Lagrangian Method

Solve a constrained problem using the method of multipliers.

In [ ]:
# Augmented Lagrangian for equality constraints
def f_aug(x):
    return 0.5 * (x[0] ** 2 + x[1] ** 2)


def h_aug(x):
    return x[0] + x[1] - 1


def augmented_lagrangian(x, lam, mu):
    return f_aug(x) + lam * h_aug(x) + 0.5 * mu * h_aug(x) ** 2


def grad_aug(x, lam, mu):
    dh = np.array([1.0, 1.0])
    return np.array([x[0], x[1]]) + (lam + mu * h_aug(x)) * dh


x = np.array([0.0, 0.0])
lam = 0.0
mu = 1.0
print('Augmented Lagrangian method:')
for k in range(20):
    for _ in range(10):
        g = grad_aug(x, lam, mu)
        H = np.eye(2) + mu * np.outer([1, 1], [1, 1])
        x = x - la.solve(H, g)
    lam = lam + mu * h_aug(x)
    if k % 5 == 0 or k == 19:
        print(f'k={k:2d}: x=({x[0]:.6f}, {x[1]:.6f}), h(x)={h_aug(x):.2e}, lambda={lam:.6f}')
ok = abs(h_aug(x)) < 1e-8 and np.allclose(x, [0.5, 0.5], atol=1e-6)
print('PASS' if ok else 'FAIL', '- converged to x* = (0.5, 0.5)')

---

## 13. Handling Non-Convex Constraints

Solve a problem with non-convex constraints using SQP.

In [ ]:
# === 13.1 SQP for non-convex problem ===
# Problem: min (x1-2)^2 + (x2-1)^2 s.t. x1*x2 >= 1 (non-convex feasible set)

def f_nc(x): return (x[0]-2)**2 + (x[1]-1)**2
def g_nc(x): return 1 - x[0]*x[1]  # g(x) <= 0 means x1*x2 >= 1

def grad_f_nc(x): return np.array([2*(x[0]-2), 2*(x[1]-1)])
def grad_g_nc(x): return np.array([-x[1], -x[0]])

x0 = np.array([2.0, 2.0])
constraints = {'type': 'ineq', 'fun': lambda x: x[0]*x[1] - 1,
               'jac': lambda x: np.array([x[1], x[0]])}

res = opt.minimize(f_nc, x0, jac=grad_f_nc, constraints=constraints,
                   method='SLSQP', options={'maxiter': 100})

print(f'SQP for non-convex constraint x1*x2 >= 1:')
print(f'Solution: x = ({res.x[0]:.6f}, {res.x[1]:.6f})')
print(f'Objective: {res.fun:.6f}')
print(f'Constraint: x1*x2 = {res.x[0]*res.x[1]:.6f} (should be >= 1)')
print(f'Success: {res.success}')

# Here the unconstrained minimizer (2,1) is already feasible, so the non-convex constraint is inactive.
ok = res.success and np.linalg.norm(res.x - np.array([2.0, 1.0])) < 1e-3 and res.x[0]*res.x[1] >= 1-1e-6
print()
print(f"{'PASS' if ok else 'FAIL'} — the solver found the feasible unconstrained minimizer")

---

## 14. Duality Gap Analysis

Compute the duality gap for a constrained optimization problem.

In [ ]:
# === 14.1 Duality gap for QP ===
# Problem: min 0.5*x^T P x + q^T x s.t. Ax = b

P = np.array([[4.0, 1.0], [1.0, 2.0]])
q = np.array([1.0, 1.0])
A = np.array([[1.0, 1.0]])
b = np.array([1.0])

K = np.block([[P, A.T], [A, np.zeros((1, 1))]])
rhs = np.concatenate([-q, b])
sol = la.solve(K, rhs)
x_star = sol[:2]
nu_star = sol[2:]

p_star = 0.5 * x_star @ P @ x_star + q @ x_star
P_inv = np.linalg.inv(P)
d_star = -0.5 * (q + A.T @ nu_star).T @ P_inv @ (q + A.T @ nu_star) - b @ nu_star

d_star = float(d_star)
print(f'Primal optimal value: p* = {p_star:.6f}')
print(f'Dual optimal value: d* = {d_star:.6f}')
print(f'Duality gap: p* - d* = {p_star - d_star:.2e}')

ok = abs(p_star - d_star) < 1e-8
print()
print(f"{'PASS' if ok else 'FAIL'} — strong duality holds (gap = 0)")

---

## 15. Sensitivity Analysis

Analyze how the optimal value changes with constraint perturbations.

In [ ]:
# === 15.1 Sensitivity of optimal value ===
# Problem: min 0.5*||x||^2 s.t. x1 + x2 = b
# Solution: x* = (b/2, b/2), p*(b) = b^2/4
# Sensitivity: dp*/db = b/2 = lambda*

b_values = np.linspace(-2, 2, 201)
p_values = []
lambda_values = []

for b in b_values:
    x_star = np.array([b/2, b/2])
    p_star = 0.5 * np.sum(x_star**2)
    lambda_star = b/2  # Lagrange multiplier
    p_values.append(p_star)
    lambda_values.append(lambda_star)

p_values = np.array(p_values)
lambda_values = np.array(lambda_values)

# Numerical derivative
dp_db_num = np.gradient(p_values, b_values, edge_order=2)

print('Sensitivity analysis: dp*/db vs lambda*')
print(f'Max |dp*/db - lambda*|: {np.max(np.abs(dp_db_num - lambda_values)):.2e}')

ok = np.max(np.abs(dp_db_num - lambda_values)) < 1e-4
print(f"\n{'PASS' if ok else 'FAIL'} — dp*/db = lambda* (shadow price interpretation)")

In [ ]:
# === 15.2 Plot sensitivity ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(b_values, p_values, color=COLORS['primary'])
    axes[0].set_title('Optimal value vs constraint RHS')
    axes[0].set_xlabel('b')
    axes[0].set_ylabel('p*(b)')
    
    axes[1].plot(b_values, lambda_values, color=COLORS['secondary'], label='lambda*(b)')
    axes[1].plot(b_values, dp_db_num, '--', color=COLORS['neutral'], label='dp*/db (numerical)')
    axes[1].set_title('Shadow price: lambda* = dp*/db')
    axes[1].set_xlabel('b')
    axes[1].set_ylabel('Value')
    axes[1].legend()
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 16. Summary: All Methods Compared

Final comparison of constrained optimization methods on a test problem.

In [ ]:
# === 16.1 Method comparison ===
# Problem: min 0.5*||x||^2 s.t. x >= 1 (element-wise)
n = 100
x_true = np.ones(n)

def f_comp(x): return 0.5 * np.sum(x**2)
def grad_f_comp(x): return x

def proj_comp(x): return np.maximum(x, 1)

# 1. Projected GD
x_pgd = np.zeros(n)
eta = 0.5
for _ in range(500):
    x_pgd = proj_comp(x_pgd - eta * grad_f_comp(x_pgd))
err_pgd = np.linalg.norm(x_pgd - x_true)

# 2. Penalty method
x_pen = np.zeros(n)
for mu in [1, 10, 100, 1000, 10000]:
    def f_pen_mu(x): return 0.5*np.sum(x**2) + mu*np.sum(np.maximum(1-x, 0)**2)
    res = opt.minimize(f_pen_mu, x_pen, method='BFGS')
    x_pen = res.x
err_pen = np.linalg.norm(x_pen - x_true)

# 3. scipy SLSQP
constraints = [{'type': 'ineq', 'fun': lambda x: x[i] - 1} for i in range(n)]
res = opt.minimize(f_comp, np.zeros(n), jac=grad_f_comp,
                   constraints=constraints, method='SLSQP')
err_sl = np.linalg.norm(res.x - x_true)

print(f'Method comparison: min 0.5*||x||^2 s.t. x >= 1 (n={n})')
print(f'{'Method':<20} | {'||x - x*||':>12}')
print('-' * 35)
print(f"{'Projected GD':<20} | {err_pgd:>12.2e}")
print(f"{'Penalty method':<20} | {err_pen:>12.2e}")
print(f"{'scipy SLSQP':<20} | {err_sl:>12.2e}")

ok = min(err_pgd, err_pen, err_sl) < 0.01
print(f"\n{'PASS' if ok else 'FAIL'} — at least one method converged")

---

## 17. KKT Verification for QP

Verify KKT conditions for a general quadratic program.

In [ ]:
from scipy.optimize import minimize
P = np.array([[4.0, 1.0], [1.0, 2.0]])
q = np.array([1.0, 1.0])
G = np.array([[-1.0, 0.0], [0.0, -1.0], [1.0, 1.0]])
h = np.array([0.0, 0.0, 2.0])
A = np.array([[1.0, 1.0]])
b = np.array([1.0])
def qp_obj(x): return 0.5 * x @ P @ x + q @ x
def qp_grad(x): return P @ x + q
ineq_c = [{'type': 'ineq', 'fun': lambda x, i=i: h[i] - G[i] @ x} for i in range(len(h))]
eq_c = [{'type': 'eq', 'fun': lambda x: A @ x - b}]
res = minimize(qp_obj, np.array([0.5, 0.5]), jac=qp_grad,
               constraints=ineq_c + eq_c, method='SLSQP')
x_star = res.x
print(f'QP solution: x* = {x_star}')
print(f'Primal feasibility (Gx <= h): {np.all(G @ x_star <= h + 1e-6)}')
print(f'Primal feasibility (Ax = b): {np.allclose(A @ x_star, b)}')
ok = np.all(G @ x_star <= h + 1e-6) and np.allclose(A @ x_star, b)
print(f"\n{'PASS' if ok else 'FAIL'} — primal feasibility satisfied")

---

## 18. L1 Ball Projection

Project onto $\mathcal{B}_1(r) = \{\mathbf{x} : \lVert \mathbf{x} \rVert_1 \leq r\}$.

In [ ]:
def proj_l1_ball(v, radius=1.0):
    if np.linalg.norm(v, 1) <= radius:
        return v
    u = np.sort(np.abs(v))[::-1]
    cssv = np.cumsum(u) - radius
    ind = np.arange(len(v)) + 1
    cond = u - cssv / ind > 0
    rho = ind[cond][-1]
    theta = cssv[cond][-1] / rho
    return np.sign(v) * np.maximum(np.abs(v) - theta, 0)
v = np.array([0.8, -0.6, 0.4, -0.3, 0.5])
r = 1.0
p = proj_l1_ball(v, r)
print(f'Input: {v}, ||v||_1 = {np.linalg.norm(v, 1):.4f}')
print(f'Projected: {p}, ||p||_1 = {np.linalg.norm(p, 1):.10f}')
ok = np.linalg.norm(p, 1) <= r + 1e-10
print(f"\n{'PASS' if ok else 'FAIL'} — L1 projection correct")

---

## 19. Constrained Rosenbrock

Minimize Rosenbrock subject to $\lVert \mathbf{x} \rVert_2 \leq 1.5$.

In [ ]:
def rosenbrock(x): return (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
def rosenbrock_grad(x):
    dx = -2*(1-x[0]) - 400*x[0]*(x[1]-x[0]**2)
    dy = 200*(x[1]-x[0]**2)
    return np.array([dx, dy])
def proj_ball(x, r):
    norm = np.linalg.norm(x)
    return x if norm <= r else r * x / norm
Delta = 1.5
x = np.array([0.0, 0.0])
eta = 0.001
for _ in range(10000):
    x = proj_ball(x - eta * rosenbrock_grad(x), Delta)
print(f'Solution: ({x[0]:.6f}, {x[1]:.6f}), ||x||={np.linalg.norm(x):.6f}')
print(f'f(x) = {rosenbrock(x):.6f}')
ok = np.linalg.norm(x) <= Delta + 1e-6
print(f"\n{'PASS' if ok else 'FAIL'} — constraint satisfied")

---

## 20. Dual Decomposition

Solve $\min f_1(x_1) + f_2(x_2)$ s.t. $x_1 = x_2$ via dual ascent.

In [ ]:
def f1(x): return (x - 1)**2
def f2(x): return 2 * (x - 3)**2
def x1_star(lam): return 1 - lam/2
def x2_star(lam): return 3 + lam/4
lam = 0.0
for _ in range(100):
    lam = lam + 0.1 * (x1_star(lam) - x2_star(lam))
x1_opt, x2_opt = x1_star(lam), x2_star(lam)
print(f'lambda = {lam:.6f}, x1 = {x1_opt:.6f}, x2 = {x2_opt:.6f}')
print(f'Consensus error: |x1-x2| = {abs(x1_opt-x2_opt):.2e}')
x_analytical = 7/3
ok = abs(x1_opt - x_analytical) < 0.01 and abs(x2_opt - x_analytical) < 0.01
print(f"\n{'PASS' if ok else 'FAIL'} — consensus at x = 7/3")

---

## 21. Non-Negative Matrix Factorization

Factorize $V \approx WH$ with $W, H \geq 0$ using projected GD.

In [ ]:
np.random.seed(42)
m, n, k = 50, 40, 5
V = np.abs(np.random.randn(m, n))
W = np.abs(np.random.randn(m, k))
H = np.abs(np.random.randn(k, n))
lr = 0.001
obj_hist = []
for t in range(500):
    W = np.maximum(W - lr * (W @ H - V) @ H.T, 0)
    H = np.maximum(H - lr * W.T @ (W @ H - V), 0)
    obj_hist.append(0.5 * np.linalg.norm(W @ H - V, 'fro')**2)
print(f'NMF: Initial={obj_hist[0]:.2f}, Final={obj_hist[-1]:.2f}')
print(f'Min(W)={W.min():.6f}, Min(H)={H.min():.6f}')
ok = W.min() >= -1e-10 and H.min() >= -1e-10 and obj_hist[-1] < obj_hist[0]
print(f"\n{'PASS' if ok else 'FAIL'} — NMF converged with non-negative factors")

---

## Summary

Additional demonstrations:

17. **KKT verification** — complete KKT check for general QP
18. **L1 ball projection** — sparse projection via simplex algorithm
19. **Constrained Rosenbrock** — projected GD on non-convex objective
20. **Dual decomposition** — distributed optimization via dual ascent
21. **NMF** — non-negative matrix factorization via projected GD